# Congested Traffic Four-Experiment DQN Test

This notebook runs a clean 2x2 comparison on the same congested traffic setup:

- Baseline DQN
- Attention DQN
- Baseline DQN + TTC safety reward
- Attention DQN + TTC safety reward

Only the attention model and TTC safety reward are toggled. Adaptive speed control, lane-context observation, and lane-change safety penalties are disabled here so the comparison stays focused.

## Imports And Paths

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "congested_traffic"
RESULTS_SUBDIR = "ct4"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "dqn" / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_ROOT / "src" / "deep_learning" / "DQN", PROJECT_ROOT / "notebooks" / "_shared"]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import dqn_notebook_utils

dqn_notebook_utils = importlib.reload(dqn_notebook_utils)
from dqn_notebook_utils import (
    build_dqn_args,
    build_env_config,
    evaluate_saved_model,
    load_dqn_backend,
    train_and_display,
)

print("Project root:", PROJECT_ROOT)
print("Results dir:", RESULTS_DIR)

## Shared Congested Setup

In [ ]:
environment_profile = "structured_baseline"

congested_environment_overrides = {
    "lanes_count": 3,
    "vehicles_count": 30,
    "duration": 40,
    "ego_spacing": 1.8,
    "vehicles_density": 1.2,
    "simulation_frequency": 15,
    "policy_frequency": 3,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
}

congested_observation_config = {
    "vehicles_count": 12,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
    "see_behind": True,
}

action_config = {
    "type": "DiscreteMetaAction",
}

base_reward_config = {
    "collision_reward": -5.0,
    "right_lane_reward": 0.03,
    "high_speed_reward": 0.25,
    "lane_change_reward": -0.02,
    "normalize_reward": False,
}

speed_config = {
    "reward_speed_range": [15.0, 28.0],
}

# Surrounding drivers vary from conservative to aggressive, but this is shared by all four runs.
congested_driver_aggressiveness_config = {
    "enabled": True,
    "distribution": "uniform",
    "min_score": 0.0,
    "max_score": 100.0,
    "fixed_score": None,
    "normal_mean": 50.0,
    "normal_std": 20.0,
    "conservative": {
        "target_speed": 18.0,
        "acc_max": 4.0,
        "comfort_acc_max": 2.0,
        "comfort_acc_min": -4.0,
        "delta": 4.5,
        "time_wanted": 2.4,
        "distance_wanted": 14.0,
        "politeness": 0.8,
        "lane_change_min_acc_gain": 0.8,
        "lane_change_max_braking_imposed": 1.0,
        "lane_change_delay": 1.5,
    },
    "aggressive": {
        "target_speed": 30.0,
        "acc_max": 7.0,
        "comfort_acc_max": 5.5,
        "comfort_acc_min": -6.5,
        "delta": 3.5,
        "time_wanted": 0.6,
        "distance_wanted": 6.0,
        "politeness": 0.0,
        "lane_change_min_acc_gain": 0.05,
        "lane_change_max_braking_imposed": 3.5,
        "lane_change_delay": 0.5,
    },
}

safety_ttc_flow_reward_config = {
    "enabled": True,
    "ttc_safe_threshold": 4.0,
    "ttc_target": 6.0,
    "ttc_cap": 10.0,
    "low_ttc_penalty_weight": 2.0,
    "max_low_ttc_penalty": 2.0,
    "safe_ttc_bonus_weight": 0.05,
    "max_safe_ttc_bonus": 0.10,
    "lag_penalty_weight": 0.08,
    "speed_tolerance": 2.0,
    "max_lag_penalty": 0.5,
    "rear_ttc_pressure": 5.0,
    "rear_pressure_floor": 0.15,
    "flow_radius": 120.0,
    "lanes": "ego_and_adjacent",
}

attention_config = {
    "features_dim": 64,
    "attention_heads": 2,
    "attention_dropout": 0.0,
    "presence_feature_idx": 0,
    "embedding_arch": "64,64",
    "net_arch": "64,64",
}

timesteps = 20_000
saved_model_eval_episodes = 1000
eval_episodes_during_training = 10
n_envs = min(4, os.cpu_count() or 1)
seed = 42
train_freq = 4
gradient_steps = 4
saved_model_eval_seed = seed + 10_000
saved_model_eval_name = f"eval{saved_model_eval_episodes}"

hyperparameters = {
    "learning_rate": 2.5e-4,
    "buffer_size": 50_000,
    "learning_starts": 2_000,
    "batch_size": 64,
    "gamma": 0.95,
    "target_update_interval": 1_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.40,
    "exploration_initial_eps": 1.0,
    "exploration_final_eps": 0.05,
    "progress_every": 1_000,
    "verbose": 0,
}

display(pd.DataFrame({"shared_training_config": pd.Series({
    "timesteps": timesteps,
    "saved_model_eval_episodes": saved_model_eval_episodes,
    "eval_episodes_during_training": eval_episodes_during_training,
    "n_envs": n_envs,
    "results_dir": str(RESULTS_DIR),
})}))

## Runner Helpers

In [ ]:
completed_experiments: dict[str, dict] = {}


def make_congested_env_config(*, safety_reward: bool = False) -> dict:
    return build_env_config(
        profile_name=environment_profile,
        profile_overrides=congested_environment_overrides,
        observation=congested_observation_config,
        action=action_config,
        reward=base_reward_config,
        speed=speed_config,
        adaptive_longitudinal={"enabled": False},
        rear_flow={"enabled": False},
        traffic_flow_reward={"enabled": False},
        safety_ttc_flow_reward=safety_ttc_flow_reward_config if safety_reward else {"enabled": False},
        driver_aggressiveness=congested_driver_aggressiveness_config,
        driver_aggressiveness_observation={"enabled": False},
        ttc_observation={"enabled": False},
        lane_change_safety={"enabled": False},
    )


def run_four_way_experiment(
    *,
    run_name: str,
    label: str,
    backend_module: str,
    attention: bool = False,
    safety_reward: bool = False,
    seed_offset: int = 0,
) -> dict:
    trainer, _, _, results_dir, default_device = load_dqn_backend(
        backend_module=backend_module,
        notebook_subdir="congested_traffic_policy",
        results_subdir=RESULTS_SUBDIR,
    )
    env_config = make_congested_env_config(safety_reward=safety_reward)
    args = build_dqn_args(
        results_dir=results_dir,
        run_name=run_name,
        timesteps=timesteps,
        eval_episodes=eval_episodes_during_training,
        seed=seed + seed_offset,
        num_envs=n_envs,
        device=default_device,
        hyperparameters=hyperparameters,
        extra=attention_config if attention else None,
    )
    summary = train_and_display(trainer, args, env_config, label)
    saved_eval_summary = evaluate_saved_model(
        trainer,
        summary_path=results_dir / run_name / "summary.json",
        env_config=env_config,
        episodes=saved_model_eval_episodes,
        seed=saved_model_eval_seed + seed_offset,
        name=saved_model_eval_name,
        label=label,
    )
    output = {
        "run_name": run_name,
        "label": label,
        "backend_module": backend_module,
        "attention": bool(attention),
        "safety_reward": bool(safety_reward),
        "summary": summary,
        "saved_eval_summary": saved_eval_summary,
    }
    completed_experiments[run_name] = output
    return output


def build_comparison_table(outputs: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for run_name, output in outputs.items():
        row = {
            "run_name": run_name,
            "label": output["label"],
            "attention": output["attention"],
            "safety_reward": output["safety_reward"],
        }
        for record in output["saved_eval_summary"].get("metric_summary", []):
            metric = (
                record.get("metric", "")
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("/", "_per_")
                .lower()
            )
            row[f"{metric}_mean"] = record.get("mean")
            row[f"{metric}_std"] = record.get("std")
        rows.append(row)
    return pd.DataFrame(rows)

## Four Experiments

In [ ]:
EXPERIMENT_SPECS = [
    {
        "run_name": "base20k",
        "label": "Baseline DQN",
        "backend_module": "elurant_dqn",
        "attention": False,
        "safety_reward": False,
        "seed_offset": 0,
    },
    {
        "run_name": "attn20k",
        "label": "Attention DQN",
        "backend_module": "attention_dqn",
        "attention": True,
        "safety_reward": False,
        "seed_offset": 100,
    },
    {
        "run_name": "safe20k",
        "label": "Baseline DQN + Safety Reward",
        "backend_module": "elurant_dqn",
        "attention": False,
        "safety_reward": True,
        "seed_offset": 200,
    },
    {
        "run_name": "attn_safe20k",
        "label": "Attention DQN + Safety Reward",
        "backend_module": "attention_dqn",
        "attention": True,
        "safety_reward": True,
        "seed_offset": 300,
    },
]

display(pd.DataFrame(EXPERIMENT_SPECS))

## Run All

In [ ]:
for spec in EXPERIMENT_SPECS:
    print(f"\n=== Running {spec['label']} ===", flush=True)
    run_four_way_experiment(**spec)

comparison_df = build_comparison_table(completed_experiments)
comparison_path = RESULTS_DIR / "comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Comparison saved to:", comparison_path)
display(comparison_df.round(3))

## Compare Current Session

In [ ]:
if not completed_experiments:
    print("Run the experiment cell above first.")
else:
    comparison_df = build_comparison_table(completed_experiments)
    display(comparison_df.round(3))